<a href="https://colab.research.google.com/github/acapodanno/Large-Language-Model/blob/main/rag_final_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Setup delle dipendenze: Installare e configurare chromadb, sentence-transformers, rank-bm25 e langchain per l'ecosistema completo**

In [193]:
!pip install langchain langchain-community rank_bm25 faiss-cpu chromadb langchain-openai langchain-classic

In [194]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader_txt = TextLoader("./solar_system.txt",encoding="utf-8")
doc_txt = loader_txt.load()


#text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50,separators=["\n\n", "\n", ". ", " ", "","**"])
#txt_chunks = text_splitter.split_documents(doc_txt)


### **Architettura modulare: Progettare classi base astratte per chunking, permettendo estensibilità futura con nuove strategie**



In [195]:
from abc import ABC, abstractmethod
from typing import List
from langchain_core.documents import Document

class BaseChunker(ABC):
    def __init__(self, chunk_size: int, chunk_overlap: int):
      self.chunk_size = chunk_size
      self.chunk_overlap = chunk_overlap

    @abstractmethod
    def chunk_documents(self, documents: List[Document]) -> List[Document]:
      pass

    def __repr__(self) -> str:
       return super().__repr__()


In [208]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
class RecursiveChunckerStrategy(BaseChunker):
  def __init__(self, chunk_size: int = 1000, chunk_overlap: int = 200):
        super().__init__(chunk_size, chunk_overlap)
        self.name="RecursiveCharacterTextSplitter"
        self._splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.chunk_size,
            chunk_overlap=self.chunk_overlap,
            separators=["\n\n", "\n", ". ", " ", ""]
        )
  def chunk_documents(self, documents: List[Document]) -> List[Document]:
    print(f"Esecuzione chunking con {self.name}...")
    return self._splitter.split_documents(doc_txt)

from langchain_text_splitters import TokenTextSplitter
class TokenBasedStrategy(BaseChunker):
  def __init__(self, chunk_size: int = 1000, chunk_overlap: int = 200):
        super().__init__(chunk_size, chunk_overlap)
        self.name="TokenTextSplitter"
        self.chunk_documents = TokenTextSplitter(
            chunk_size=self.chunk_size,
            chunk_overlap=self.chunk_overlap,
        )
  def chunk_documents(self, documents: List[Document]) -> List[Document]:
    print(f"Esecuzione chunking con {self.name}...")
    return self._splitter.split_documents



In [206]:
rcs = RecursiveChunckerStrategy(chunk_size=500, chunk_overlap=50)
txt_chunks = rcs.chunk_documents(doc_txt)

Esecuzione chunking con RecursiveCharacterTextSplitter...


### Vector database: Configurare ChromaDB con HNSW indexing per ricerche efficienti su grandi dataset

In [198]:
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
import chromadb
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

#embeddings = HuggingFaceEmbeddings(model_name="paraphrase-multilingual-MiniLM-L12-v2")

client = chromadb.PersistentClient(path="./nuovo_db_vettoriale_v2")

vectorstore = Chroma.from_documents(
    documents=txt_chunks,
    embedding=embeddings,
    client=client,
    collection_name="documenti_astronomia"
)
dense_retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 3,
    })

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [199]:
dense_retriever.invoke("Qual'é il pianet piú piccolo ??")

[Document(metadata={'source': './solar_system.txt'}, page_content='Il sole é il pianeta piu grande!!!\n\n\nIl Sole\nIl Sole è il cuore del nostro sistema. È una stella di classe spettrale G2V (spesso chiamata nana gialla) ed è composto principalmente da idrogeno (circa 73%) ed elio (circa 25%). Costituisce da solo il 99,86% di tutta la massa del Sistema Solare. La sua energia, generata dalla fusione nucleare nel nucleo, è la forza motrice che permette la vita sulla Terra e determina il clima e il meteo di tutti i pianeti.'),
 Document(metadata={'source': './solar_system.txt'}, page_content='Il sole é il pianeta piu grande!!!\n\n\nIl Sole\nIl Sole è il cuore del nostro sistema. È una stella di classe spettrale G2V (spesso chiamata nana gialla) ed è composto principalmente da idrogeno (circa 73%) ed elio (circa 25%). Costituisce da solo il 99,86% di tutta la massa del Sistema Solare. La sua energia, generata dalla fusione nucleare nel nucleo, è la forza motrice che permette la vita sulla

### **Preprocessing intelligente: Sviluppare pipeline di pulizia testo, tokenizzazione e normalizzazione per BM25**

In [200]:
from langchain_community.retrievers import BM25Retriever
import re



def _tokenize(text: str) -> List[str]:
  """Tokenizzazione semplice per BM25"""
  text = text.lower()
  tokens = re.findall(r'\b\w+\b', text)
  return [t for t in tokens if len(t) > 2]

retriever_bm25 = BM25Retriever.from_documents(
    documents=txt_chunks,
    preprocess_func=_tokenize
)
retriever_bm25.k = 3
retriever_bm25.invoke("sOl3")


[Document(metadata={'source': './solar_system.txt'}, page_content="* La Nube di Oort: Si ipotizza che esista un guscio sferico immenso, situato ai confini estremi dell'influenza gravitazionale del Sole (fino a 100.000 Unità Astronomiche di distanza). Questa nube è considerata la culla delle comete di lungo periodo."),
 Document(metadata={'source': './solar_system.txt'}, page_content="* La Fascia di Kuiper: Una regione a forma di ciambella ricca di corpi ghiacciati e pianeti nani. Qui si trova Plutone, declassato da nono pianeta a pianeta nano nel 2006. Plutone ha una geologia sorprendentemente attiva e montagne di ghiaccio d'acqua. Altri pianeti nani famosi qui includono Eris, Haumea e Makemake."),
 Document(metadata={'source': './solar_system.txt'}, page_content='6. Oltre Nettuno: Il Regno del Ghiaccio\nIl Sistema Solare non finisce con Nettuno. Oltre la sua orbita si estendono vastissime regioni inesplorate.')]

In [201]:
hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, retriever_bm25],
    weights=[0.7, 0.3]
)

In [202]:
print("Ricerca 1: Testiamo una parola chiave/nome esatto")
query_1 = "Cos'è Sagittarius A* e dove si trova?"
risultati_1 = hybrid_retriever.invoke(query_1)
print(f"Risultato: {risultati_1[0].page_content}\n")

Ricerca 1: Testiamo una parola chiave/nome esatto
Risultato: Il Centro Galattico
Nel cuore della Via Lattea, a circa 26.000 anni luce dalla Terra, si trova una regione estremamente densa e attiva. Al centro esatto risiede *Sagittarius A\, un buco nero supermassiccio con una massa pari a circa 4 milioni di volte quella del nostro Sole. Nonostante la sua immensa gravità, il buco nero non sta "risucchiando" l'intera galassia; le stelle e i sistemi planetari orbitano attorno a questo centro di massa proprio come i pianeti orbitano attorno al Sole.



In [203]:
print("Ricerca 2: Testiamo il significato (senza usare le parole esatte)")
query_2 = "Cosa fa da scudo al nostro pianeta(Terra) impedendo che l'aria voli via nello spazio?"
risultati_2 = hybrid_retriever.invoke(query_2)
print(f"Risultato: {risultati_2}\n")

Ricerca 2: Testiamo il significato (senza usare le parole esatte)
Risultato: [Document(metadata={'source': './solar_system.txt'}, page_content='Il quinto pianeta è un colosso: Giove ha una massa due volte e mezzo superiore a quella di tutti gli altri pianeti messi insieme. È composto principalmente da idrogeno ed elio, in modo simile al Sole. È famoso per le sue spettacolari bande di nubi e per la Grande Macchia Rossa, una tempesta anticiclonica più grande della Terra che imperversa da secoli'), Document(metadata={'source': './solar_system.txt'}, page_content='* Il Campo Magnetico: Il nucleo terrestre, composto da ferro e nichel fusi in movimento, genera una potente magnetosfera. Questo "scudo" devia il vento solare (un flusso di particelle cariche provenienti dal Sole) che altrimenti strapperebbe via la nostra atmosfera.'), Document(metadata={'source': './solar_system.txt'}, page_content='1. La Via Lattea: La Nostra Casa Cosmica\nLa Via Lattea è la galassia in cui si trova il nostro S

In [204]:
print("Ricerca 3: Testiamo l'azione combinata")
query_3 = "Da quali gas è composta la stella di classe spettrale G2V?"
risultati_3 = hybrid_retriever.invoke(query_3)
print(f"Risultato: {risultati_3[0].page_content}")

Ricerca 3: Testiamo l'azione combinata
Risultato: Il Sole
Il Sole è il cuore del nostro sistema. È una stella di classe spettrale G2V (spesso chiamata nana gialla) ed è composto principalmente da idrogeno (circa 73%) ed elio (circa 25%). Costituisce da solo il 99,86% di tutta la massa del Sistema Solare. La sua energia, generata dalla fusione nucleare nel nucleo, è la forza motrice che permette la vita sulla Terra e determina il clima e il meteo di tutti i pianeti.


In [205]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

system_prompt = (
    "Sei un assistente esperto in astronomia. "
    "Usa ESCLUSIVAMENTE i seguenti frammenti di contesto per rispondere alla domanda dell'utente. "
    "Se la risposta non è presente nel contesto, rispondi semplicemente che non lo sai, non inventare nulla. "
    "Scrivi in un italiano chiaro, discorsivo e naturale."
    "\n\n"
    "CONTESTO TROVATO NEL DATABASE:\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm, prompt)

rag_chain = create_retrieval_chain(hybrid_retriever, question_answer_chain)
domanda = "soLe"

print("Sto pensando (cercando nei documenti e generando la risposta)...")
risposta_finale = rag_chain.invoke({"input": domanda})

print("\n🤖 RISPOSTA DELL'IA:")
print("-" * 50)
print(risposta_finale["answer"])
print("-" * 50)

Sto pensando (cercando nei documenti e generando la risposta)...

🤖 RISPOSTA DELL'IA:
--------------------------------------------------
Il Sole è una stella di classe spettrale G2V, spesso chiamata nana gialla. È composto principalmente da idrogeno ed elio e costituisce il 99,86% della massa del Sistema Solare. La sua energia, generata dalla fusione nucleare nel nucleo, è fondamentale per la vita sulla Terra e influisce sul clima e sul meteo di tutti i pianeti.
--------------------------------------------------
